In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import matplotlib.dates as mdates
import seaborn as sns
from FinMind.data import DataLoader
api = DataLoader()

In [2]:
import pandas as pd
import numpy as np

class DCA_With_Dip_Buying_Optimized:
    """
    優化版：定期定額 + 階梯式跌幅加碼 (與前次價格相比)
    """
    def __init__(self, base_monthly_investment=10000, fee_min=1, fee_rate=0.001425, dip_config={
                     'small':  {'threshold': -0.03, 'multiplier': 1.0,  'active': True},
                     'medium': {'threshold': -0.05, 'multiplier': 1.5,  'active': True},
                     'big':    {'threshold': -0.10, 'multiplier': 10.0, 'active': True}
                 }):
        self.base_monthly = base_monthly_investment
        self.fee_min = fee_min        # 最低手續費 (1元)
        self.fee_rate = fee_rate      # 原始手續費率 (通常 0.001425)
        self.investment_log = []
        self.shares_owned = 0
        self.total_invested = 0
        self.total_fees = 0           # 累計手續費
        self.dividend_log = []
        self.last_dip_price = None
        self.dip_config = dip_config

    def get_actual_trading_dates(self, df, day_of_month=15):
        """
        找出每個月『大於或等於』指定日期的第一個實際交易日
        """
        # 提取資料中所有的年份與月份
        df_temp = df.copy()
        df_temp['year_month'] = df_temp['date'].dt.to_period('M')
        unique_months = df_temp['year_month'].unique()
        
        actual_dates = []
        for ym in unique_months:
            # 設定該月的目標日期 (例如 2023-01-15)
            # 使用 min(day_of_month, 該月最後一天) 以免 31 號出錯
            target_date = pd.Timestamp(year=ym.year, month=ym.month, day=1) + pd.offsets.Day(day_of_month - 1)
            
            # 在 df 中尋找大於等於 target_date 的第一個日期
            trading_day = df[df['date'] >= target_date]['date'].min()
            
            # 如果找到了且該日期確實屬於同一個月份（避免跨月過頭），則加入
            if pd.notnull(trading_day):
                actual_dates.append(trading_day)
        
        return pd.DatetimeIndex(actual_dates)

    def apply_strategy(self, df, div_df, invest_day=15):
        """
        df: 股價資料 (date, close)
        div_df: 股利資料
        invest_day: 每月定期定額日期
        """
        if not div_df.empty:
            div_df['CashExDividendTradingDate'] = pd.to_datetime(div_df['CashExDividendTradingDate'])
        df['date'] = pd.to_datetime(df['date'])
        self.monthly_dates = self.get_actual_trading_dates(df, day_of_month=invest_day)
    

        for i in range(len(df)):
            current_date = df['date'].iloc[i]
            current_price = df['close'].iloc[i]
            is_monthly_day = current_date in self.monthly_dates
            investment_amount = 0
            trigger_reasons = []

            # 1. 定期定額 (每月的固定動作)
            if is_monthly_day:
                investment_amount += self.base_monthly
                self.total_fees += 2 if self.base_monthly > 10000 else 1
                trigger_reasons.append("定期定額")
                self.last_dip_price = current_price
            elif self.last_dip_price is not None:
                # 計算「當前價」與「上次加碼價」的差距
                price_diff_ratio = (current_price - self.last_dip_price) / self.last_dip_price
                conf = self.dip_config
                dip_trigger = None

                if conf['big']['active'] and price_diff_ratio <= conf['big']['threshold']:
                    dip_trigger = f"大跌加碼({conf['big']['threshold']*100}%)"
                    investment_amount = self.base_monthly * conf['big']['multiplier']
                elif conf['medium']['active'] and price_diff_ratio <= conf['medium']['threshold']:
                    dip_trigger = f"中跌加碼({conf['medium']['threshold']*100}%)"
                    investment_amount = self.base_monthly * conf['medium']['multiplier']
                elif conf['small']['active'] and price_diff_ratio <= conf['small']['threshold']:
                    dip_trigger = f"小跌加碼({conf['small']['threshold']*100}%)"
                    investment_amount = self.base_monthly * conf['small']['multiplier']

                if dip_trigger:
                    self.total_fees += max(self.fee_min, int(investment_amount * self.fee_rate))
                    trigger_reasons.append(dip_trigger)
                    self.last_dip_price = current_price

            # 3. 執行買入
            if investment_amount > 0:
                shares_bought = investment_amount / current_price
                self.shares_owned += shares_bought
                self.total_invested += investment_amount
                self.investment_log.append({
                    'date': current_date,
                    'price': current_price,
                    'amount': investment_amount,
                    'shares': shares_bought,
                    'trigger': " + ".join(trigger_reasons),
                    'total_shares': self.shares_owned
                })

            # 4. 股利處理 # 判断是否是除息日
            if not div_df.empty:
                day_div = div_df[div_df['CashExDividendTradingDate']== current_date]
                # 计算当前持有的股份应得的现金股利
                # 除息日当天持有的股份有权领取股利
                if not day_div.empty and self.shares_owned > 0:
                    cash_dividend = day_div['CashEarningsDistribution'].iloc[0]
                    dividend_income = self.shares_owned * cash_dividend
                    dividend_income = self.shares_owned * cash_dividend
                    self.dividend_log.append({
                        'date': current_date,
                        'dividend_per_share': cash_dividend,
                        'dividend_income': dividend_income,
                        'shares_owned': self.shares_owned
                    })

        investment_df = pd.DataFrame(self.investment_log)
        dividend_df = pd.DataFrame(self.dividend_log) if self.dividend_log else pd.DataFrame()
        return investment_df, dividend_df
    
    def get_performance(self, current_price):
        """
        计算绩效，包含现金股利收益。
        
        Returns:
        --------
        tuple: (current_value, total_return, return_rate, total_dividend)
        """
        if self.shares_owned == 0:
            return 0, 0, 0, 0
        
        # 计算股票市值
        current_value = self.shares_owned * current_price
        
        # 计算累计现金股利总额
        total_dividend = sum([log['dividend_income'] for log in self.dividend_log]) if hasattr(self, 'dividend_log') else 0
        
        # 计算总回报
        # 总回报 = 当前股票市值 + 累计收到的现金股利 - 总投入本金 + 手續費
        total_return = (current_value + total_dividend) - (self.total_invested + self.total_fees)
        
        # 计算总回报率
        if self.total_invested > 0:
            return_rate = (total_return / self.total_invested) * 100
        else:
            return_rate = 0
        
        return current_value, total_return, return_rate, total_dividend

In [3]:
today = datetime.today() #datetime.strptime('2025-04-30', "%Y-%m-%d")
start_date = (today - relativedelta(years=3)).strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")
print('start date', start_date)
print('end date', end_date)

ticker = '006208'
df = api.taiwan_stock_daily(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)

div_df = api.taiwan_stock_dividend(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)
div_df = div_df[['CashExDividendTradingDate', 'CashEarningsDistribution']]

2026-02-10 19:49:56.460 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 006208


start date 2023-02-10
end date 2026-02-10


2026-02-10 19:49:57.265 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockDividend, data_id: 006208


In [ ]:
# 執行策略
cfg = {
    'small':  {'threshold': -0.03, 'multiplier': 1.0,  'active': False},
    'medium': {'threshold': -0.05, 'multiplier': 1.5,  'active': False},
    'big':    {'threshold': -0.10, 'multiplier': 10.0, 'active': False}
}
strategy = DCA_With_Dip_Buying_Optimized(base_monthly_investment=10000, dip_config=cfg)
investment_df, dividend_df = strategy.apply_strategy(df, div_df, 3)

current_price = df['close'].iloc[-1]
current_value, total_return, return_rate, total_dividend = strategy.get_performance(current_price)

print(f"總投資金額: {strategy.total_invested:,.0f}元")
print(f"總手續費: {strategy.total_fees:,.0f} 元")
print(f"累计现金股利: {total_dividend:.2f} 元")
print(f"目前持有市值: {current_value:,.0f}元")
print(f"總報酬: {total_return:,.0f}元")
print(f"報酬率: {return_rate:.2f}%")
print(f"持有股數: {strategy.shares_owned:.2f}股")

# 显示投资记录
print("\n=== 最近10次投资记录 ===")
print(investment_df.tail(100) if len(investment_df) > 10 else investment_df)

# 显示股利记录（如果存在）
if not dividend_df.empty:
    print("\n=== 现金股利记录 ===")
    print(dividend_df)
else:
    print("\n=== 现金股利记录 ===")
    print("期间内无现金股利发放")

總投資金額: 510,000元
總手續費: 233 元
累计现金股利: 31120.93 元
目前持有市值: 948,746元
總報酬: 469,634元
報酬率: 92.09%
持有股數: 5427.61股

=== 最近10次投资记录 ===
         date   price   amount      shares      trigger  total_shares
0  2023-02-10   70.00  10000.0  142.857143         定期定額    142.857143
1  2023-03-03   69.00  10000.0  144.927536         定期定額    287.784679
2  2023-04-06   70.05  10000.0  142.755175         定期定額    430.539854
3  2023-04-25   67.60  10000.0  147.928994  小跌加碼(-3.0%)    578.468848
4  2023-05-03   67.85  10000.0  147.383935         定期定額    725.852783
5  2023-06-05   72.90  10000.0  137.174211         定期定額    863.026994
6  2023-07-03   75.60  10000.0  132.275132         定期定額    995.302127
7  2023-08-04   73.60  10000.0  135.869565         定期定額   1131.171692
8  2023-08-21   71.30  10000.0  140.252454  小跌加碼(-3.0%)   1271.424146
9  2023-09-04   73.20  10000.0  136.612022         定期定額   1408.036168
10 2023-09-22   70.90  10000.0  141.043724  小跌加碼(-3.0%)   1549.079892
11 2023-10-03   71.45  10000.0  139.

In [17]:
pd.set_option('display.max_rows', None)
investment_df

,date,price,amount,shares,trigger,total_shares
0,2023-02-07,523.0,20000,38.240918,定期定額,38.240918
1,2023-03-03,516.0,20000,38.759690,定期定額,77.000608
2,2023-04-06,530.0,20000,37.735849,定期定額,114.736457
3,2023-05-03,496.0,20000,40.322581,定期定額,155.059037
4,2023-06-05,555.0,20000,36.036036,定期定額,191.095073
5,2023-07-03,579.0,20000,34.542314,定期定額,225.637388
6,2023-08-04,554.0,20000,36.101083,定期定額,261.738471
7,2023-09-04,557.0,20000,35.906643,定期定額,297.645114
8,2023-10-03,529.0,20000,37.807183,定期定額,335.452297
9,2023-11-03,549.0,20000,36.429872,定期定額,371.882169
